In [ ]:
import flax.linen as nn
import optax
import h5py as hf
import matplotlib.pyplot as plt
import znnl as nl
import numpy as np
from znnl.data.decision_boundary import (
    DecisionBoundaryGenerator,
    circle,
    linear_boundary,
)

# Dask support
from dask.distributed import Client
from dask_jobqueue import SLURMCluster

In [ ]:
ds_size=100

In [ ]:
class Generator(nl.data.DataGenerator):
    def __init__(self, ds_size, module, dimension: int = 2):
        self.ds_size = ds_size

        model = nl.models.FlaxModel(
            flax_module=module, 
            optimizer=optax.adam(0.01), 
            input_shape=(1, dimension),
            batch_size=50
        )

        train_inputs = np.random.uniform(size=(ds_size, dimension))
        train_labels = model(train_inputs)

        test_inputs = np.random.uniform(size=(ds_size, dimension))
        test_labels = model(test_inputs)

        validation_inputs = np.random.uniform(size=(ds_size, dimension))
        validation_labels = model(validation_inputs)

        self.train_ds = {
            "inputs": train_inputs,
            "targets": train_labels,
        }
        self.test_ds = {
            "inputs": test_inputs,
            "targets": test_labels,
        }
        self.validation_ds = {
            "inputs": validation_inputs,
            "targets": validation_labels,
        }

In [ ]:
def train(index: int):
    """
    Run the experiment.
    """

    class Network(nn.Module):
        """
        Perceptron network.
        """

        @nn.compact
        def __call__(self, x):
            """
            Call method for the network
            """
            x = nn.Dense(3, use_bias=True)(x)
            x = nn.relu(x)
            x = nn.Dense(3, use_bias=True)(x)
            x = nn.relu(x)
            x = nn.Dense(1, use_bias=True)(x)
            return x
        
    generator = Generator(ds_size, Network(), dimension=2)
    model = nl.models.FlaxModel(
        flax_module=Network(), 
        optimizer=optax.adam(0.01), 
        input_shape=(1, 2),
        batch_size=50
    )
    # Prepare the recorders
    train_recorder = nl.training_recording.JaxRecorder(
        name=f"two-layer-3/train_recorder_{index}",
        loss=True,
        entropy=True,
        trace=True,
        accuracy=True,
        magnitude_variance=True,
        update_rate=1,
    )
    test_recorder = nl.training_recording.JaxRecorder(
        name=f"two-layer-3/test_recorder_{index}", 
        loss=True, 
        accuracy=True, 
        update_rate=1,
    )
    train_recorder.instantiate_recorder(data_set=generator.train_ds)
    test_recorder.instantiate_recorder(data_set=generator.test_ds)

    trainer = nl.training_strategies.SimpleTraining(
        model=model,
        loss_fn=nl.loss_functions.MeanPowerLoss(order=2),
        accuracy_fn=nl.accuracy_functions.LabelAccuracy(),
        recorders=[train_recorder, test_recorder],
    )

    _ = trainer.train_model(
        train_ds=generator.train_ds,
        test_ds=generator.test_ds,
        batch_size=128,
        epochs=5000,
    )


## Deployment

In [ ]:
cluster = SLURMCluster(
    cores=1,
    processes=1,
    job_cpu=30,
    memory="64GB",
    queue="single",
    walltime="18:00:00",
    log_directory=f'./two-layer-3/dask-logs',
    job_script_prologue=["module load devel/cuda/12.1"],
    job_extra_directives=["--gres=gpu:2"]
)
cluster.adapt(minimum_jobs=0, maximum_jobs=20)
client = Client(cluster)

In [ ]:
indice_blocks = np.linspace(1, 20, 20, dtype=int)

results = client.map(train, indice_blocks)


In [ ]:
results